# 01 · Select fires

Picks 30 wildfires in the western US (2015–2023), 6 per region, plus the 2024 Wapiti Fire as our local case study, and writes `fires.csv`.

Rules (from the roadmap):
- Wildfires only, ignited 2015–2023. Before 2015 there is no HLS before the fire; after 2023 there isn't enough recovery yet.
- Skip fires that burned again later, or that burned in the 10 years before. A second fire breaks the recovery signal.
- Mix vegetation (forest, shrubland, grassland) and severity (low to high) inside each region.

Run it top to bottom, in Colab or locally in VS Code. It takes ~10–20 min, most of it in step 2.
Outputs: `fires.csv` (the picks) and `candidates.csv` (every fire that passed the filters, in case we want to swap one).

## 0. Settings

In [ ]:
import os

# Your Earth Engine (Google Cloud) project ID. Leave it empty to read it from
# ee_project.txt at the repo root, which git ignores, so each of us can use our own.
PROJECT_ID = ""
if not PROJECT_ID:
    for path in ("ee_project.txt", "../ee_project.txt"):
        if os.path.exists(path):
            PROJECT_ID = open(path).read().strip()
assert PROJECT_ID, "Put your Earth Engine project ID in ee_project.txt at the repo root"

# Colab -> your Google Drive; local (VS Code) -> data/fires inside the repo
IN_COLAB = os.path.exists("/content")
OUT_DIR = "/content/drive/MyDrive/cs497/data/fires" if IN_COLAB else "../data/fires"

YEARS = (2015, 2023)
MIN_ACRES = 10_000        # one 224x224 chip at 30 m is ~11,000 acres
FIRES_PER_REGION = 6
SEED = 497                # same seed -> same picks

REGIONS = {
    "California": ["CA"],
    "Pacific Northwest": ["OR", "WA"],
    "Rockies": ["MT", "WY", "CO", "ID"],
    "Southwest": ["AZ", "NM"],
    "Great Basin": ["NV", "UT"],
}

MAX_REBURN = 0.05         # max share of the fire that burned again later
MAX_PRIOR_BURN = 0.05     # max share that burned in the PRIOR_YEARS before
PRIOR_YEARS = 10
MIN_VEG_SHARE = 0.6       # forest + shrub + grass inside the perimeter
MIN_PICK_ACRES = 20_000   # only pick fires big enough for several chips
MIN_MOD_HIGH = 0.25       # at least 25% of the mapped perimeter must be moderate or high severity
# Grassland fires are smaller and mostly low severity in MTBS, so they get looser limits.
# Without this, no grassland fire makes it in.
GRASS_MIN_PICK_ACRES = 15_000
GRASS_MIN_MOD_HIGH = 0.15
MAX_NODATA = 0.2          # MTBS non-mapping area (clouds, water) inside the perimeter

# Event_IDs we want in or out no matter what.
# The first 15 fires (3 per region, picked Nov 4) are already exported, so they stay.
FORCE_INCLUDE = [
    "CA4114212301620210731",
    "CA3612312160220200819",
    "CA3415811689620150617",
    "OR4310412287520210802",
    "OR4552312036720160721",
    "OR4237012386020180716",
    "CO3957210726620200810",
    "MT4817010962220170827",
    "MT4755110759720210722",
    "AZ3286511020120210610",
    "NM3624410503720220417",
    "AZ3341510931220200606",
    "NV4152811742720180705",
    "UT3771811283720170617",
    "NV4004911989320160729",
]
# Holiday Farm (OR, 2020): lots of salvage logging after the fire, which breaks the recovery signal
FORCE_EXCLUDE = ["OR4417212223120200908"]

CASE_STUDY = {"name": "WAPITI", "year": 2024, "state": "ID"}

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import ee

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Candidate fires from MTBS

MTBS perimeters, wildfires only, big enough for at least one chip, in our 11 states.
We drop single-scene assessments (`Asmnt_Type` ending in "(SS)") because those had no pre-fire image.

In [ ]:
MTBS = ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
SEVERITY = ee.ImageCollection("USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1")
NLCD = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD")

STATE_TO_REGION = {s: r for r, states in REGIONS.items() for s in states}
ATTRS = ["Event_ID", "Incid_Name", "Ig_Date", "BurnBndAc", "BurnBndLat", "BurnBndLon", "Asmnt_Type", "state"]


def to_millis(year, month=1, day=1):
    return int(pd.Timestamp(year=year, month=month, day=day, tz="UTC").timestamp() * 1000)


def in_year(fc, year):
    return (fc.filter(ee.Filter.gte("Ig_Date", to_millis(year)))
              .filter(ee.Filter.lt("Ig_Date", to_millis(year + 1))))


def add_state(f):
    # Event_ID starts with the state, e.g. "CA3858812244220170613"
    return f.set("state", ee.String(f.get("Event_ID")).slice(0, 2))


candidates = (
    MTBS.filter(ee.Filter.eq("Incid_Type", "Wildfire"))
    .filter(ee.Filter.gte("Ig_Date", to_millis(YEARS[0])))
    .filter(ee.Filter.lt("Ig_Date", to_millis(YEARS[1] + 1)))
    .filter(ee.Filter.gte("BurnBndAc", MIN_ACRES))
    .filter(ee.Filter.inList("Asmnt_Type", ["Initial", "Extended"]))
    .map(add_state)
    .filter(ee.Filter.inList("state", list(STATE_TO_REGION)))
)
print("Candidate fires:", candidates.size().getInfo())

## 2. Stats inside each perimeter

For every candidate:
- **Vegetation**: NLCD land cover from the last epoch before the fire (forest 41–43, shrub 52, grass 71).
- **Severity**: MTBS severity classes for the fire year.
- **Reburn**: share of the perimeter inside any MTBS fire that started later.
- **Prior burn**: share inside any MTBS fire in the 10 years before.

Results are cached in `OUT_DIR`, so a rerun skips this step.

In [ ]:
# Last NLCD epoch before the fire year, so land cover is pre-fire
NLCD_EPOCH = {2015: "2013", 2016: "2013", 2017: "2016", 2018: "2016", 2019: "2016",
              2020: "2019", 2021: "2019", 2022: "2019", 2023: "2019", 2024: "2019"}
HIST_SCALE = 30    # keep land cover / severity classes at their native 30 m
SHARE_SCALE = 90   # coarser is fine for burned shares
DAY_MS = 86_400_000


def histogram(img, band, geom):
    return img.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(), geometry=geom,
        scale=HIST_SCALE, maxPixels=1e10, tileScale=4,
    ).get(band)


def burned_share(fires, geom):
    # 1 inside any of the given fires, 0 outside -> mean = share of geom that burned
    img = ee.Image(0).paint(fires, 1).rename("burned")
    return img.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=geom,
        scale=SHARE_SCALE, maxPixels=1e10, tileScale=4,
    ).get("burned")


def image_table(col):
    """id, year and study area of every image in a collection."""
    rows = []
    for im in col.getInfo()["features"]:
        p = im.get("properties", {})
        t = p.get("system:time_start")
        years = re.findall(r"(?:19|20)\d\d", im["id"])
        year = pd.Timestamp(t, unit="ms").year if t is not None else (int(years[-1]) if years else None)
        rows.append({"id": im["id"], "year": year, "area": str(p.get("study_area", "")).upper()})
    return pd.DataFrame(rows)


NLCD_TABLE = image_table(NLCD)
SEV_TABLE = image_table(SEVERITY)
print("NLCD images:", NLCD_TABLE["id"].str.split("/").str[-1].tolist())
print("MTBS severity images:", len(SEV_TABLE), "| last ones:", SEV_TABLE["id"].str.split("/").str[-1].tolist()[-4:])


def nlcd_image(year):
    t = NLCD_TABLE[NLCD_TABLE["year"] == int(NLCD_EPOCH[year])]
    if len(t) != 1:
        raise ValueError(f"NLCD epoch {NLCD_EPOCH[year]}: found {t['id'].tolist()}")
    return ee.Image(t["id"].iloc[0]).select("landcover")


def severity_image(year):
    t = SEV_TABLE[SEV_TABLE["year"] == year]
    if len(t) > 1:  # one mosaic per study area (CONUS, AK, HI, PR)
        t = t[(t["area"] == "CONUS") | t["id"].str.upper().str.contains("CONUS")]
    if len(t) != 1:
        raise ValueError(f"MTBS severity {year}: found {t['id'].tolist()}")
    return ee.Image(t["id"].iloc[0]).select("Severity")


def stats_for_year(fc, year):
    nlcd, sev = nlcd_image(year), severity_image(year)

    def one(f):
        g = f.geometry()
        ig = ee.Number(f.get("Ig_Date"))
        others = MTBS.filterBounds(g).filter(ee.Filter.neq("Event_ID", f.get("Event_ID")))
        later = others.filter(ee.Filter.gt("Ig_Date", ig))
        before = (others.filter(ee.Filter.lt("Ig_Date", ig))
                        .filter(ee.Filter.gte("Ig_Date", ig.subtract(PRIOR_YEARS * 365 * DAY_MS))))
        stats = {
            "nlcd": histogram(nlcd, "landcover", g),
            "severity": histogram(sev, "Severity", g),
            "reburn": burned_share(later, g),
            "prior": burned_share(before, g),
        }
        return ee.Feature(ee.Feature(None, stats).copyProperties(f, ATTRS))

    return fc.map(one)


RETRYABLE = ("timed out", "timeout", "memory", "too many", "deadline", "capacity")


def fetch(fc, fn, depth=0):
    """getInfo of fn(fc). If Earth Engine times out or runs out of memory,
    split fc in half and try again. Any other error is a real bug, so raise it."""
    try:
        return [f["properties"] for f in fn(fc).getInfo()["features"]]
    except ee.EEException as e:
        if not any(s in str(e).lower() for s in RETRYABLE):
            raise
        n = fc.size().getInfo()
        if n <= 1 or depth > 8:
            print("   skipped", n, "fire(s):", str(e)[:120])
            return []
        items = fc.toList(n)
        half = n // 2
        return (fetch(ee.FeatureCollection(items.slice(0, half)), fn, depth + 1)
                + fetch(ee.FeatureCollection(items.slice(half)), fn, depth + 1))

In [ ]:
FOREST, SHRUB, GRASS = (41, 42, 43), (52,), (71,)


def clean_hist(h):
    # EE returns keys as strings ("41" or "41.0")
    return {int(float(k)): v for k, v in (h or {}).items()}


def share(h, classes, total):
    return sum(h.get(c, 0) for c in classes) / total if total else np.nan


def veg_type(forest, shrub, grass):
    shares = {"forest": forest, "shrubland": shrub, "grassland": grass}
    best = max(shares, key=shares.get)
    return best if shares[best] >= 0.5 else "mixed"


def severity_class(moderate, high):
    if high >= 0.3:
        return "high"
    if moderate + high < 0.3:
        return "low"
    return "moderate"


def summarize(row):
    nlcd, sev = clean_hist(row["nlcd"]), clean_hist(row["severity"])
    n_nlcd, n_sev = sum(nlcd.values()), sum(sev.values())
    n_burn = sum(sev.get(c, 0) for c in (1, 2, 3, 4))  # unburned/low .. high
    out = {
        "forest": share(nlcd, FOREST, n_nlcd),
        "shrub": share(nlcd, SHRUB, n_nlcd),
        "grass": share(nlcd, GRASS, n_nlcd),
        "sev_unburned_low": share(sev, (1,), n_burn),
        "sev_low": share(sev, (2,), n_burn),
        "sev_moderate": share(sev, (3,), n_burn),
        "sev_high": share(sev, (4,), n_burn),
        "sev_nodata": share(sev, (6,), n_sev),
        "reburn": row["reburn"] or 0.0,
        "prior": row["prior"] or 0.0,
    }
    out["veg_share"] = out["forest"] + out["shrub"] + out["grass"]
    out["veg_type"] = veg_type(out["forest"], out["shrub"], out["grass"])
    out["severity"] = severity_class(out["sev_moderate"], out["sev_high"])
    return pd.Series(out)


def to_table(rows):
    df = pd.DataFrame(rows)
    df = pd.concat([df.drop(columns=["nlcd", "severity", "reburn", "prior"]),
                    df.apply(summarize, axis=1)], axis=1)
    ig = pd.to_datetime(df["Ig_Date"], unit="ms", utc=True)
    df["ignition_date"] = ig.dt.strftime("%Y-%m-%d")
    df["year"] = ig.dt.year
    df["region"] = df["state"].map(STATE_TO_REGION)
    for c in ("BurnBndLat", "BurnBndLon"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.rename(columns={
        "Event_ID": "mtbs_id", "Incid_Name": "name", "BurnBndAc": "acres",
        "BurnBndLat": "lat", "BurnBndLon": "lon", "Asmnt_Type": "assessment",
    }).drop(columns=["Ig_Date"])

In [ ]:
stats_path = os.path.join(OUT_DIR, "candidate_stats.csv")

if os.path.exists(stats_path):
    stats = pd.read_csv(stats_path)
    print("Loaded cached stats:", len(stats), "fires")
else:
    rows = []
    for year in range(YEARS[0], YEARS[1] + 1):
        out = fetch(in_year(candidates, year), lambda fc, y=year: stats_for_year(fc, y))
        print(year, "->", len(out), "fires")
        rows += out
    stats = to_table(rows)
    stats.to_csv(stats_path, index=False)

stats.head()

## 3. Filter and pick

Keep the clean fires, then in each region pick 3 that differ as much as possible in vegetation type, severity and year. Vegetation type counts most, then severity, then year.

In [ ]:
def is_clean(df):
    grass = df["veg_type"] == "grassland"
    min_acres = np.where(grass, GRASS_MIN_PICK_ACRES, MIN_PICK_ACRES)
    min_mod_high = np.where(grass, GRASS_MIN_MOD_HIGH, MIN_MOD_HIGH)
    return ((df["reburn"] <= MAX_REBURN)
            & (df["prior"] <= MAX_PRIOR_BURN)
            & (df["veg_share"] >= MIN_VEG_SHARE)
            & (df["sev_nodata"] <= MAX_NODATA)
            & (df["acres"] >= min_acres)
            & ((df["sev_moderate"] + df["sev_high"]) >= min_mod_high)
            & ~df["mtbs_id"].isin(FORCE_EXCLUDE))


def pick_diverse(df, n, seed=0):
    """Greedy: each new pick adds a vegetation type, severity or year we don't have yet."""
    pool = df.sample(frac=1, random_state=seed)  # shuffle so ties are random but repeatable
    chosen = []
    while len(chosen) < n and len(pool):
        vegs = {r["veg_type"] for r in chosen}
        sevs = {r["severity"] for r in chosen}
        years = {r["year"] for r in chosen}
        new_veg = ~pool["veg_type"].isin(vegs) & (pool["veg_type"] != "mixed")  # prefer clear types
        score = (3 * new_veg  # vegetation type counts most
                 + 2 * ~pool["severity"].isin(sevs)
                 + 1 * ~pool["year"].isin(years))
        i = score.astype(int).idxmax()  # first best in shuffled order
        chosen.append(pool.loc[i])
        pool = pool.drop(index=i)
    return pd.DataFrame(chosen)


def select_fires(stats, per_region, seed):
    clean = stats[is_clean(stats)]
    forced = stats[stats["mtbs_id"].isin(FORCE_INCLUDE)]
    picks = []
    for region in REGIONS:
        have = forced[forced["region"] == region]
        rest = clean[(clean["region"] == region) & ~clean["mtbs_id"].isin(have["mtbs_id"])]
        picks += [have, pick_diverse(rest, max(per_region - len(have), 0), seed)]
    return clean, pd.concat(picks)


clean, picked = select_fires(stats, FIRES_PER_REGION, SEED)
print(f"{len(stats)} candidates -> {len(clean)} clean -> {len(picked)} picked")
print(clean.groupby(["region", "veg_type"]).size().unstack(fill_value=0))

## 4. Add the Wapiti Fire (case study)

MTBS goes through 2024, so it should be there. If it isn't, we add it by hand from the Wapiti RECOVER package.

In [ ]:
wapiti_fc = (in_year(MTBS, CASE_STUDY["year"])
             .filter(ee.Filter.stringContains("Incid_Name", CASE_STUDY["name"]))
             .map(add_state)
             .filter(ee.Filter.eq("state", CASE_STUDY["state"])))
try:
    wapiti_rows = fetch(wapiti_fc, lambda fc: stats_for_year(fc, CASE_STUDY["year"]))
except ValueError as e:  # e.g. no 2024 severity mosaic yet
    print(e)
    wapiti_rows = []

if wapiti_rows:
    wapiti = to_table(wapiti_rows)
    wapiti["region"] = "Rockies"
    wapiti["role"] = "case_study"
    print(wapiti[["name", "mtbs_id", "ignition_date", "acres", "veg_type", "severity", "reburn"]])
else:
    wapiti = pd.DataFrame()
    print("Wapiti not found in MTBS -> add it by hand from the RECOVER package.")

## 5. Save

In [ ]:
COLUMNS = ["name", "mtbs_id", "year", "ignition_date", "state", "region", "veg_type", "severity",
           "acres", "lat", "lon", "forest", "shrub", "grass", "sev_low", "sev_moderate", "sev_high",
           "reburn", "prior", "assessment", "role"]

picked = picked.assign(role="study")
fires = pd.concat([picked, wapiti], ignore_index=True)
fires = fires[COLUMNS].round(3)

fires.to_csv(os.path.join(OUT_DIR, "fires.csv"), index=False)
clean.assign(role="candidate").reindex(columns=COLUMNS).round(3).sort_values(["region", "year"]) \
     .to_csv(os.path.join(OUT_DIR, "candidates.csv"), index=False)

fires[["name", "year", "state", "region", "veg_type", "severity", "acres", "role"]]

In [ ]:
print(fires.groupby("region")[["veg_type", "severity"]].agg(lambda s: ", ".join(sorted(s))))
print()
print("Vegetation:", fires["veg_type"].value_counts().to_dict())
print("Severity:  ", fires["severity"].value_counts().to_dict())
print("Years:     ", sorted(fires["year"].unique()))

## 6. Quick map

In [ ]:
import folium

COLORS = dict(zip(REGIONS, ["red", "blue", "green", "orange", "purple"]))
m = folium.Map(location=[40, -113], zoom_start=5, tiles="Esri.WorldGrayCanvas")
for _, r in fires.iterrows():
    folium.CircleMarker(
        [r["lat"], r["lon"]], radius=7, fill=True, fill_opacity=0.9,
        color="black" if r["role"] == "case_study" else COLORS.get(r["region"], "gray"),
        tooltip=f"{r['name']} ({r['year']}) · {r['veg_type']} · {r['severity']} · {int(r['acres']):,} ac",
    ).add_to(m)
os.makedirs("../figures", exist_ok=True)
m.save("../figures/fires_map.html")
m

In [ ]:
import matplotlib.pyplot as plt

states = (ee.FeatureCollection("TIGER/2018/States")
          .filter(ee.Filter.inList("STUSPS", [s for group in REGIONS.values() for s in group]))
          .map(lambda f: f.simplify(2000))
          .getInfo())

fig, ax = plt.subplots(figsize=(10, 9))
for f in states["features"]:
    g = f["geometry"]
    # simplify can return a GeometryCollection with stray 2-point lines, so keep only the polygons
    for part in g.get("geometries", [g]):
        polygons = {"Polygon": [part["coordinates"]], "MultiPolygon": part["coordinates"]}.get(part["type"], [])
        for polygon in polygons:
            for ring in polygon:
                ax.plot(*zip(*ring), color="lightgray", linewidth=0.8, zorder=1)

for _, r in fires.iterrows():
    case_study = r["role"] == "case_study"
    ax.scatter(r["lon"], r["lat"], s=r["acres"] / 1000, zorder=3,
               marker="*" if case_study else "o",
               color="black" if case_study else COLORS[r["region"]])
    ax.annotate(r["name"], (r["lon"], r["lat"]), xytext=(5, 3), textcoords="offset points", fontsize=7)

for region, color in COLORS.items():
    ax.scatter([], [], color=color, label=region)
ax.legend(loc="lower left")
ax.set_title("Selected fires (MTBS, 2015-2023)")
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
fig.savefig("../figures/fires_map.png", dpi=200, bbox_inches="tight")
plt.show()

## Notes

- Regions are by state. Idaho goes in the Rockies even though the south of the state is Great Basin.
- MTBS perimeters only go through 2024, so a reburn in 2025–2026 would not be caught here. Step 2 can double-check with the HLS time series (a second sudden NBR drop).
- To swap a fire: put its `mtbs_id` from `candidates.csv` in `FORCE_INCLUDE` (or the one to drop in `FORCE_EXCLUDE`) and rerun sections 3–5. The stats are cached, so it's fast.